# Sample 3: Autonomous Flower Video with Text Overlays

This notebook demonstrates the autonomous agent creating a video with:
- Flower images analyzed by vision AI
- Text description overlays generated with Pillow
- Multi-track timeline with overlay compositing

In [1]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../../")))

from dotenv import load_dotenv
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
from openai import AsyncOpenAI, OpenAI

# Load environment variables
load_dotenv()

async_client = AsyncOpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
)

openai_client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
)

chat_completion_service = OpenAIChatCompletion(
    service_id="my-service-id",
    ai_model_id="openai/gpt-4o-mini",
    async_client=async_client
)

In [2]:
from openagv import AssetBin, SKLoopExecutor, UserInstruction, OTIOTimeline
from openagv.modules.vision import ORVisionAnalyzer
from openagv.modules.textcard import TextCardGenerator

# Define the instruction for autonomous agent
instruct = UserInstruction(
    """Create a video showcasing flowers with descriptive text overlays.
    
    For each flower image:
    1. First analyze the image to get a description
    2. Generate a lower-third text overlay with the flower name and a short description
    3. Add the flower to the main timeline track (3 seconds each)
    4. Add the text overlay at the same time position on an overlay track
    
    Make sure every flower has both a background clip and a text overlay.
    Sort flowers by how vibrant/colorful they appear."""
)

# Setup AssetBin and add flower images
ab = AssetBin()
ab.add_wildcard("../assets/*.jpg")

# Initialize modules
vision_analyzer = ORVisionAnalyzer(client=openai_client, model='openai/gpt-4o-mini')
textcard = TextCardGenerator(output_dir="./text_cards", width=1920, height=1080, font_size=36)
timeline = OTIOTimeline(width=1920, height=1080, fps=30, name="Flowers with Overlays")

# Setup Executor with all modules
ex = SKLoopExecutor(
    ab, 
    instruct, 
    chat_completion=chat_completion_service, 
    uses=[vision_analyzer, textcard, timeline], 
    debug=True
)

# Execute autonomously
await ex.start()

[INFO] Starting execution...
[DEBUG] Chat History: 2 messages
[INFO] Advancing to step: AssetBin.list_possible_unanalyzed
[DEBUG] Invoking AssetBin.list_possible_unanalyzed with args: {}
[DEBUG] Result from AssetBin.list_possible_unanalyzed: Asset: ../assets\aarn-giri-3tYZjGSBwbk-unsplash.jpg (ID: bbd4872365cab6c1e5c099270e047a514cc3f15d6902ccdaac88bb01e444b958) -> Pending: ORVisionAnalyzer (openai/gpt-4o-mini)
Asset: ../assets\andrew-small-EfhCUc_fjrU-unsplash.jpg (ID: b0ea3882b9aba6470a20dd2f05b77caa18b354226020e9e1ed9d2bb20f4ca806) -> Pending: ORVisionAnalyzer (openai/gpt-4o-mini)
Asset: ../assets\evie-s-w1JE5duY62M-unsplash.jpg (ID: 6786e06d6f727b5b8cf39ecb41c9f61ca4fe2763ff8ea066d7ce25c3f8845690) -> Pending: ORVisionAnalyzer (openai/gpt-4o-mini)
Asset: ../assets\olia-gozha-9A_peGrSbZc-unsplash.jpg (ID: b8ef5dd5efbf3c851db53edc34f38eac7f17d3000dfe478ded645d7174498796) -> Pending: ORVisionAnalyzer (openai/gpt-4o-mini)
Asset: ../assets\sergey-shmidt-koy6FlCCy5s-unsplash.jpg (ID: 2530

In [3]:
# Check timeline status
print(timeline.get_summary())
print(f"\nOverlay tracks: {timeline.get_overlay_tracks()}")

Timeline 'Flowers with Overlays' - Main track: 6 items, 18.0s

Overlay tracks: []


In [4]:
# Nudge if needed to ensure all flowers have overlays
await ex.nudge(UserInstruction(
    "Please verify that every flower in the timeline has a corresponding text overlay. "
    "If any are missing, add them now."
))

[INFO] Nudged with: Please verify that every flower in the timeline has a corresponding text overlay. If any are missing, add them now.
[DEBUG] Chat History: 34 messages
[INFO] Advancing to step: OTIOTimeline.get_summary
[DEBUG] Invoking OTIOTimeline.get_summary with args: {}
[DEBUG] Result from OTIOTimeline.get_summary: Timeline 'Flowers with Overlays' - Main track: 6 items, 18.0s
[INFO] Advancing to step: ChecklistManager.set_checklist
[DEBUG] Invoking ChecklistManager.set_checklist with args: {items='Verify text overlays for each flower in the timeline.
Add missing overlays if any found.'}
[DEBUG] Result from ChecklistManager.set_checklist: Checklist initialized with 2 tasks.
[INFO] Advancing to step: ChecklistManager.get_remaining_tasks
[DEBUG] Invoking ChecklistManager.get_remaining_tasks with args: {}
[DEBUG] Result from ChecklistManager.get_remaining_tasks: Remaining Tasks:
1. Verify text overlays for each flower in the timeline.
2. Add missing overlays if any found.
[INFO] Adva

In [5]:
# Save timeline
timeline.to_otio_file('flowers_overlay.otio')
print("Timeline saved to flowers_overlay.otio")

Timeline saved to flowers_overlay.otio


In [6]:
# Render the final video with overlays
from openagv.renderer import FfmpegOTIORenderer

renderer = FfmpegOTIORenderer()
renderer.set_otio(timeline)
renderer.validate()
renderer.render('flowers_with_overlays.mp4')

Executing FFmpeg render...
Render complete: flowers_with_overlays.mp4
